# Entraînement du détecteur de cartes (GPU gratuit Colab)

1. Menu **Exécution → Modifier le type d'exécution → GPU (T4)**
2. Le dernier modèle du dépôt (`detector/weights/best.pt`) est affiné automatiquement : rien à déposer.
3. **Exécution → Tout exécuter**. Gardez l'onglet ouvert et empêchez la mise en veille du PC.
4. À la fin, `cards.onnx`, `meta.json` et `best.pt` se téléchargent : déposer les deux premiers dans `app/model/` sur GitHub (branche `claude/adoring-gates-5by66o`), garder `best.pt` pour la prochaine fois.

In [ ]:
!nvidia-smi -L || echo 'PAS DE GPU : Exécution > Modifier le type d’exécution > GPU'

In [ ]:
BRANCH = 'claude/adoring-gates-5by66o'
!rm -rf cardcounter && git clone -q --depth 1 -b $BRANCH https://github.com/hugohismans/cardcounter.git
%cd cardcounter/detector
!pip -q install 'ultralytics>=8.4' opencv-python-headless onnx onnxslim onnxruntime

Paramètres : plus d'images / d'epochs = meilleur modèle, mais plus long.

In [ ]:
import os
# On affine le dernier modèle : best.pt déposé dans /content s'il y en a un, sinon celui du dépôt
# (detector/weights/best.pt). Sans aucun des deux : entraînement depuis zéro.
start = next((p for p in ('/content/best.pt', 'weights/best.pt') if os.path.exists(p)), None)
FINETUNE = start is not None
MODEL = start or 'yolo26n.pt'
TRAIN, VAL, BATCH = 6000, 500, 32
EPOCHS = 15 if FINETUNE else 40
print(f'Affinage de {start}' if FINETUNE else 'Entraînement depuis zéro', '-', EPOCHS, 'epochs')
os.environ.update(TRAIN=str(TRAIN), VAL=str(VAL), EPOCHS=str(EPOCHS), BATCH=str(BATCH), MODEL=MODEL)
!./train_all.sh

In [ ]:
from google.colab import files
files.download('../app/model/cards.onnx')
files.download('../app/model/meta.json')
files.download('runs/cards/weights/best.pt')  # utile pour affiner plus tard